# 🎯 What is actually in this file

**Module 1 · Session 03, part 1**

**The brief.** You have joined a team that builds playlists. Someone upstairs has decided that
popularity ought to be predictable, and wants to know which musical qualities make a track
popular so the label can commission more of them.

You have 32,833 rows of Spotify data: ten audio measurements per track, a genre, and a
popularity score from 0 to 100. The question is answerable. Whether the answer is the one they
want is a different matter, and finding that out is the whole job today.

Part 1 looks at the file. Part 2 tests what turns up.

## 🔍 The five questions

Work in this order on any table you are handed. This notebook goes through them once, on a
file you already know from session 02.

| | Question | Where to look |
|---|---|---|
| 1 | How big is it? | `.shape`, `.info()` |
| 2 | What is missing? | `.isna().sum()`, and then the values that are present but wrong |
| 3 | What shape is each column, and is anything impossible? | `.describe()`, then plot it |
| 4 | What groups are there, and how big? | `.value_counts()` |
| 5 | What moves with what? | `.corr()`, plus a picture |

Every one of those is a single line of code, which is why they get skipped.


In [ ]:
# Setup. Same file as last week, plus one style block so every chart here matches.
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import HTML, IFrame

# One theme for the whole notebook, so no chart below needs styling of its own.
# The colours are chosen to stay distinguishable with colour-blindness.
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#8b8a85"
sns.set_theme(style="whitegrid", palette=[BLUE, ORANGE, GREY],
              rc={"figure.dpi": 110, "grid.color": "#ececea", "font.size": 10,
                  "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left"})

# The file lives in the class repository. pandas reads a URL exactly like a file.
URL = "https://raw.githubusercontent.com/aaubs/ds-master/codex/m1-pandas-2026/data/M1_2026/spotify_songs.csv"

songs = pd.read_csv(URL).rename(columns={
    "track_name": "title", "track_artist": "artist",
    "track_popularity": "popularity", "playlist_genre": "genre"})
tracks = songs.drop_duplicates("track_id")   # one row per song instead of one per placement


def listen(rows, *extra_columns):
    """Show rows with a clickable Spotify link. track_id is a real Spotify ID."""
    out = rows[["title", "artist", *extra_columns]].round(3)
    out["listen"] = "https://open.spotify.com/track/" + rows["track_id"]
    return HTML(out.to_html(render_links=True, escape=False, index=False))


def play(rows):
    """Play each row in the notebook. Also takes a bare Spotify track id.
    If the player does not render in your environment, the listen() links always work."""
    ids = [rows] if isinstance(rows, str) else rows["track_id"]
    for track_id in ids:
        display(IFrame(f"https://open.spotify.com/embed/track/{track_id}", width="100%", height=80))


print(f"Rows (a track on a playlist): {len(songs):,}")
print(f"Distinct songs:               {len(tracks):,}")


Two frames out of one file. `songs` has a row every time a track appears on a playlist, so a
song on five playlists is in there five times. `tracks` has one row per song.

Which one you want depends on what you are claiming. A typical song? Use `tracks`. What is on
these playlists? Use `songs`. Get this wrong and you have not made a coding error, you have
answered a different question.

## 1 and 2. How big, and what is missing


In [ ]:
print("Rows and columns:", songs.shape)
songs.info()


In [ ]:
missing = songs.isna().sum()
display(missing[missing > 0].to_frame("missing"))


Twenty-three columns, five rows with no title, artist or album name, and nothing missing
anywhere else. Session 02 settled what to do about that, which here is nothing.

So the popularity column is complete. That is not the same as the popularity column being
right.


In [ ]:
print("Tracks scoring exactly 0:", int(tracks["popularity"].eq(0).sum()))
print("Share of the file:      ", f"{tracks['popularity'].eq(0).mean():.1%}")

fig, ax = plt.subplots(figsize=(7.5, 3.4))
sns.histplot(tracks, x="popularity", bins=50, color=BLUE, ax=ax)
ax.set(title="Popularity, the column the brief is about", xlabel="popularity", ylabel="songs")
plt.show()


One bar is nearly five times the height of its neighbour, and it sits at zero.

Nine percent of the catalogue scores exactly 0 while almost nothing scores 1, 2 or 3. Real
scores do not behave like that. A zero could mean a track nobody streams, or it could mean
Spotify had no figure for it when the file was cut, and the column looks identical either way.

Rather than argue about it, look at who is in there.


In [ ]:
zero_scored = tracks[tracks["popularity"].eq(0)]
famous = zero_scored[zero_scored["artist"].isin(["Drake", "Taylor Swift", "Eminem", "Maroon 5"])]
display(listen(famous.head(6), "popularity"))


"Hotline Bling". Taylor Swift's "22". Four Eminem records. Play the first one and decide for
yourself whether it is a song nobody streams.


In [ ]:
# Popularity 0 according to this file, and then the other end of the scale for comparison.
play(tracks[tracks["track_id"] == "76P07ei8drjrenqtvDbefy"])   # Hotline Bling, scored 0
play(tracks.nlargest(1, "popularity"))                        # the top of the file


The second one is "Dance Monkey", the only track in the file scoring 100.

So the zeros are not a measurement of low listening. At least some of them are absence, wearing
exactly the same clothes as a real value. One more line says how widespread that is.


In [ ]:
scored_artists = set(tracks.loc[tracks["popularity"] > 0, "artist"])
also_scored = zero_scored["artist"].isin(scored_artists)

print(f"Tracks scoring 0: {len(zero_scored):,}")
print(f"...by an artist who has scored tracks elsewhere in this file: {also_scored.sum():,}")


Two thirds of them. Whatever produced these zeros, it was not a judgement about the artist.

> 🧭 **Judgement call.** You now know the column mixes two things. What you still do not know is
> which zero is which, and there is no way to find out from this file.
>
> Keep them and every average you report is dragged down by rows that are not measurements.
> Drop them and you delete nine percent of the evidence, including whichever zeros were real.
>
> Decide, say which you did, and show what changes. Part 2 puts a number on it.

## 3. What shape, and is anything impossible?


In [ ]:
display(tracks[["energy", "danceability", "valence", "tempo", "duration_ms"]].describe().round(2))


`describe` gives you eight numbers per column and hides the two things you most need. The
first is the shape.

Each orange line below is the mean of that column. Look at tempo and valence before reading on.


In [ ]:
columns = ["energy", "danceability", "valence", "tempo"]

fig, axes = plt.subplots(2, 2, figsize=(10, 5.5))
for ax, column in zip(axes.flat, columns):
    sns.histplot(tracks, x=column, bins=40, color=BLUE, ax=ax)
    ax.axvline(tracks[column].mean(), color=ORANGE, linewidth=2)   # the mean
    ax.set(title=column, xlabel="", ylabel="songs")
fig.tight_layout()
plt.show()


Four columns, four different situations, one summary statistic.

`danceability` is a single clean hump, and its mean is a fair description of a typical song.

`energy` leans hard towards the top of the range with a long tail to the left, so the mean at
0.70 sits below where most of the songs actually are.

`valence` covers the whole range with a broad plateau in the middle. The mean is 0.51, and
knowing that tells you almost nothing about any particular track.

`tempo` has two clusters, one around 95 beats per minute and one around 125, because produced
music sticks to a small number of conventional tempos. The mean lands on the taller cluster and
the second one disappears from any report that quotes only the average.


Those two humps are audible. Here is a hit from each of them, 95 beats per minute and 124.


In [ ]:
play(tracks[tracks["track_id"].isin([
    "1e9oZCCiX42nJl0AcqriVo",   # Watermelon Sugar, Harry Styles, 95 bpm
    "1DFD5Fotzgn6yYXkYsKiGs",   # Piece Of Your Heart, MEDUZA, 124 bpm
])])


Nothing about the songs forces those numbers. Producers pick tempos other producers have
picked, which is why a histogram of 28,000 tracks has gaps in it.


The second thing is whether a value is possible at all. `describe` prints the minimum and the
maximum of every column and leaves the judgement to you. Look at the `min` row again.

## ⚠️ Four thousand milliseconds is four seconds

A tempo of 0 beats per minute is not a slow song. Both of those belong to one record.


In [ ]:
display(listen(tracks.nsmallest(1, "duration_ms"), "duration_ms", "tempo", "valence", "danceability"))


In [ ]:
play(tracks.nsmallest(1, "duration_ms"))


Four seconds, no tempo, valence 0.0 and danceability 0.0, which makes it simultaneously the
shortest, the saddest and the least danceable thing in the dataset. It is not a sad song. It is
not a song.

Nobody suspected that row. It surfaced from the minimum of two columns that have nothing to do
with each other.


In [ ]:
print("Songs under one minute:  ", int((tracks["duration_ms"] < 60_000).sum()))
print("Songs with tempo exactly 0:", int((tracks["tempo"] == 0).sum()))
print("Out of:                  ", len(tracks))


Twenty-five short ones out of 28,356. A handful, not a crowd, and far too few to move a mean.

Leaving them in and saying you looked is defensible for this file. Quoting a typical song
length in a contract, you would drop them and say so. The rule changes with the claim, which is
why nobody can write it down for you in advance.

> 🧭 **Judgement call.** Ask an agent to clean this file and it will drop those rows or keep
> them, and either way it will sound sure. What it cannot know is the claim you are about to
> make, and that is the only thing the decision depends on.

## 🎧 The extremes are where the measurements give themselves away


In [ ]:
extremes = pd.concat([
    tracks.nlargest(1, "energy"), tracks.nsmallest(1, "energy"),
    tracks.nlargest(1, "instrumentalness"), tracks.nlargest(1, "valence"),
    tracks.nlargest(1, "loudness"),
])
display(listen(extremes, "genre", "energy", "valence"))


Top of the energy scale: a rainforest. Bottom: crickets near a waterfall. Most instrumental:
waves and wind. Three of the five extreme values in an audio dataset are not music, and all
three sit on a latin playlist, so they are inside the latin average that anybody would quote.


In [ ]:
play(tracks[tracks["track_id"].isin([
    "5CwOUooch74h0XarhDfAQK",   # Rain Forest and Tropical Beach Sound, energy 1.00
    "5iAB4tlYseBES4MKqgY4KG",   # Relaxing Crickets And Waterfall, energy 0.0002
    "7kigmgx2tJJsZHKaa2QC0w",   # Low Rider, War, valence 0.99
])])


Two field recordings and then, for comparison, the happiest song in the file, which the model
gets right.

Then look at the word "energy" again. It is a signal-processing measure of loudness,
density and noisiness, and it was never a claim about whether a track is exciting. Write "edm
is the most energetic genre" and you have said something about how the recordings were
mastered.

The loudest record is a 1973 Iggy Pop mix, and the happiest song, at valence 0.99, is
"Low Rider" by War. Those two the model gets right.

## 4. What groups are there, and how big?


In [ ]:
counts = songs["genre"].value_counts()

fig, ax = plt.subplots(figsize=(7, 3.2))
sns.barplot(x=counts.values, y=counts.index, color=BLUE, ax=ax)
ax.bar_label(ax.containers[0], fmt="{:,.0f}".format, padding=4)
ax.set(title="Rows per genre", xlabel="", ylabel="", xlim=(0, counts.max() * 1.15))
ax.set_xticks([])
sns.despine(bottom=True)
plt.show()


Between about 4,900 and 6,000 a side. Unusually even, and not what you normally get.

When it is uneven, a difference between a group of 40 and a group of 40,000 tells you more
about sample sizes than about the world. Print the counts next to every summary. It is a habit
rather than a technique.

## 5. What moves with what?

Here is the brief, in one line of pandas. Ten audio measurements, one popularity score, and
the question of which measurement moves with it.


In [ ]:
audio = ["danceability", "energy", "loudness", "speechiness", "acousticness",
         "instrumentalness", "liveness", "valence", "tempo", "duration_ms"]

with_popularity = tracks[audio + ["popularity"]].corr()["popularity"].drop("popularity")
display(with_popularity.sort_values(key=abs, ascending=False).round(3).to_frame("correlation with popularity"))


Nothing. The strongest relationship any audio measurement has with popularity is duration at
-0.14, and a correlation of -0.14 is close enough to nothing that you would not plan a quarter
around it. Danceability comes in at 0.05. Tempo at 0.004.

That is the answer to the brief, and it arrived in one line, well before any modelling. The
qualities Spotify measures do not tell you what people play.

For contrast, the strongest correlation anywhere in this table is between energy and loudness.


In [ ]:
print("energy against loudness:", round(tracks["energy"].corr(tracks["loudness"]), 3))


0.68, which is high, and close to a tautology. Both partly measure how much is going on in the
recording, so the number is a definition rather than a discovery.

Say which of your correlations are findings and which are definitions. It is the difference
between a result and a restatement.

## 📊 What 0.68 and -0.10 actually look like


In [ ]:
sample = tracks.sample(2_000, random_state=2026)   # 28,000 points would be a block of ink

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
sns.scatterplot(data=sample, x="loudness", y="energy", s=12, alpha=0.3,
                color=BLUE, edgecolor=None, ax=axes[0])
axes[0].set(title="r = 0.68   energy against loudness", xlabel="loudness (dB)", ylabel="energy")

sns.scatterplot(data=sample, x="energy", y="popularity", s=12, alpha=0.3,
                color=ORANGE, edgecolor=None, ax=axes[1])
axes[1].set(title="r = -0.10   popularity against energy", xlabel="energy", ylabel="popularity")

fig.tight_layout()
plt.show()


The left panel is what a strong correlation looks like, and even there the cloud is wide enough
that loudness will not predict energy for any individual song.

The right panel is what the brief is up against. There is no shape in it. Somebody who has only
seen the number -0.10 can still imagine a faint trend; nobody who has seen this panel can.

The hard line of dots along the bottom is the zero spike from earlier, showing up again in a
chart that was drawn for a different reason.

Draw the picture before you write the sentence.

## ✍️ Your turn

Pick a numeric column nobody has looked at yet. Plot it, say in one sentence whether its mean
is worth reporting, then find the most extreme song in it and play it.


In [ ]:
# One column, one plot, one sentence, one track.


### One answer, using speechiness


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
sns.histplot(tracks, x="speechiness", bins=50, color=BLUE, ax=ax)
ax.axvline(tracks["speechiness"].mean(), color=ORANGE, linewidth=2)
ax.set(title=f"speechiness (orange line = mean, {tracks['speechiness'].mean():.2f})",
       xlabel="speechiness", ylabel="songs")
plt.show()

print("median:", round(tracks["speechiness"].median(), 3))
display(listen(tracks.nlargest(2, "speechiness"), "speechiness", "genre"))
play(tracks.nlargest(1, "speechiness"))


Most songs sit near zero and a thin tail runs right, so the mean lands at 0.11 while the median
is 0.06. Reporting the mean describes almost nobody.

The extreme tracks are mostly talking, which is what the measure is for. Real records, and they
stay. A long tail is a fact about the world rather than a fault in the file, and it needs a
different response from the four-second track earlier.

## 📋 Where that leaves the brief

Five questions, one line of code each, and the headline is already in:

- 9 percent of the popularity column is a spike at zero that may not be a measurement.
- Three of the five most extreme "songs" are field recordings on a latin playlist.
- No audio measurement correlates with popularity above 0.14.

Nobody upstairs will enjoy the third one, and it is the finding. Part 2 asks whether it
survives a proper test, what the zeros do to it, and what does move popularity once the audio
columns are out of the way.
